# Clustering Practical – Student Version

**Note:** This notebook is the student version. Code cells requiring implementation are marked with `TODO`.
Please follow the instructions and fill in the missing code.

In [ ]:
import warnings
from sklearn.exceptions import ConvergenceWarning

warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=ConvergenceWarning)
warnings.filterwarnings("ignore", category=UserWarning)

## Setup & Imports
The next cell loads all core libraries plus helper functions used throughout the workshop.

In [ ]:
from __future__ import annotations

import importlib
import subprocess
import sys
from pathlib import Path
from typing import Dict, Iterable, Optional, Tuple

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from IPython.display import display

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.cluster import (
    AgglomerativeClustering,
    DBSCAN,
    HDBSCAN,
    KMeans,
    MiniBatchKMeans,
    SpectralClustering,
)
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.metrics import (
    adjusted_rand_score,
    calinski_harabasz_score,
    davies_bouldin_score,
    normalized_mutual_info_score,
    silhouette_score,
)
from sklearn.mixture import GaussianMixture
from sklearn.neighbors import NearestNeighbors

sns.set_theme(style="whitegrid", context="talk")

---
# Task 1 · Credit Portfolio Discovery Sprint (⏱️ ~25 min)

**Scenario:** Risk analytics just synced the anonymized credit-card portfolio (Kaggle’s *Credit Card Dataset for Clustering*) from Google Drive and wants proof that we can wrangle it into bank-ready personas.

**Business Context:**
- Treasury needs to know how revolving behaviour, cash advances, and payment discipline concentrate across the book of business before signing off on acquisition offers.
- Data science must demonstrate that a single CSV can stay consistent between local laptops and Colab, complete with reproducible preprocessing checkpoints.

**You'll Learn:**
- How to bulletproof Drive/Colab download flows so every student starts from the exact same file.
- Ratio engineering patterns (purchase-to-limit, payment-to-balance) that expose spend posture more clearly than raw dollars.
- How to frame “business pulse” visuals around risk levers (utilization, tenure, full-payment rate) before any clustering happens.

## 1.1 Data Access & Validation
Confirm the credit-card CSV is reachable (locally or via the Drive helper), fail loudly if it is missing, and lock a single random seed for reproducibility.

In [ ]:
if Path.cwd().name == "notebooks":
    REPO_ROOT = Path.cwd().resolve().parent
else:
    REPO_ROOT = Path.cwd().resolve()

DATA_DIR = REPO_ROOT / "data"
DATA_DIR.mkdir(exist_ok=True)
SEGMENTATION_PATH = DATA_DIR / "credit_card" / "credit_card.csv"
GDRIVE_FILE_ID = "FILE_ID"


def ensure_dataset(path: Path, drive_file_id: str) -> Path:
    """Download the dataset from Drive if it is missing locally."""
    if path.exists():
        return path
    try:
        import gdown
    except ImportError as exc:
        raise FileNotFoundError(
            "credit_card.csv not found and gdown is missing. Install it via `pip install gdown` "
            "or download the file manually from Google Drive."
        ) from exc
    path.parent.mkdir(parents=True, exist_ok=True)
    url = f"https://drive.google.com/uc?id={drive_file_id}"
    gdown.download(url, str(path), quiet=False)
    return path


SEGMENTATION_PATH = ensure_dataset(SEGMENTATION_PATH, GDRIVE_FILE_ID)
RANDOM_STATE = 42

In [ ]:
NUMERIC_BASE = [
    "BALANCE",
    "BALANCE_FREQUENCY",
    "PURCHASES",
    "ONEOFF_PURCHASES",
    "INSTALLMENTS_PURCHASES",
    "CASH_ADVANCE",
    "PURCHASES_FREQUENCY",
    "ONEOFF_PURCHASES_FREQUENCY",
    "PURCHASES_INSTALLMENTS_FREQUENCY",
    "CASH_ADVANCE_FREQUENCY",
    "CASH_ADVANCE_TRX",
    "PURCHASES_TRX",
    "CREDIT_LIMIT",
    "PAYMENTS",
    "MINIMUM_PAYMENTS",
    "PRC_FULL_PAYMENT",
    "TENURE",
]
ENGINEERED_COLUMNS = [
    "Purchase_to_Limit",
    "Cash_to_Purchases",
    "Payment_to_Balance",
    "Revolving_Ratio",
    "Installment_Share",
    "Oneoff_Share",
]
CATEGORICAL_COLUMNS = ["Tenure_Bucket", "Limit_Bucket", "Dominant_Purchase_Mode"]
REFERENCE_QUANTILE = 0.75
def load_credit_segments(path: Path) -> pd.DataFrame:
    # TODO: Implement data loading and feature engineering
    raise NotImplementedError("Implement load_credit_segments")

# credit_segments_df = load_credit_segments(SEGMENTATION_PATH)
# ... rest of preprocessing ...


## 1.2 Business Pulse Dashboards
Give stakeholders a pulse on portfolio health before clustering: utilization vs. purchases, limit tiers vs. revolving behaviour, and where payment discipline is slipping.

In [ ]:
# TODO: Implement this task
raise NotImplementedError("Student exercise")


## 3. Utility Functions
Reusable helpers streamline metric computation, clustering runs, and diagnostic plotting.

In [ ]:
from time import perf_counter


def compute_internal_metrics(
    X: np.ndarray,
    labels: np.ndarray,
    noise_label: Optional[int] = None,
) -> Dict[str, float]:
    labels = np.asarray(labels)
    mask = np.ones_like(labels, dtype=bool)
    if noise_label is not None:
        mask &= labels != noise_label
    metrics = {"silhouette": np.nan, "calinski_harabasz": np.nan, "davies_bouldin": np.nan}
    if mask.sum() > 1 and np.unique(labels[mask]).size > 1:
        X_eval = X if mask.all() else X[mask]
        y_eval = labels if mask.all() else labels[mask]
        metrics["silhouette"] = float(silhouette_score(X_eval, y_eval))
        metrics["calinski_harabasz"] = float(calinski_harabasz_score(X_eval, y_eval))
        metrics["davies_bouldin"] = float(davies_bouldin_score(X_eval, y_eval))
    return metrics


def compute_external_metrics(labels: np.ndarray, reference: Optional[pd.Series]) -> Dict[str, float]:
    results = {"adjusted_rand": np.nan, "normalized_mutual_info": np.nan}
    if reference is None:
        return results
    if len(np.unique(reference)) <= 1:
        return results
    results["adjusted_rand"] = float(adjusted_rand_score(reference, labels))
    results["normalized_mutual_info"] = float(normalized_mutual_info_score(reference, labels))
    return results


def cluster_size_summary(labels: np.ndarray, noise_label: Optional[int] = None) -> Dict[str, float]:
    labels = np.asarray(labels)
    counts = pd.Series(labels).value_counts().sort_index()
    summary = {
        "n_clusters": int(counts.index.size - (1 if noise_label is not None and noise_label in counts.index else 0)),
        "noise_fraction": float((labels == noise_label).mean()) if noise_label is not None else 0.0,
        "cluster_sizes": ", ".join(f"{int(idx)}:{int(cnt)}" for idx, cnt in counts.items()),
    }
    return summary


def evaluate_algorithm(
    name: str,
    estimator,
    X: np.ndarray,
    reference: Optional[pd.Series] = None,
    noise_label: Optional[int] = None,
) -> Tuple[pd.Series, np.ndarray, object]:
    start = perf_counter()
    if isinstance(estimator, GaussianMixture):
        estimator.fit(X)
        labels = estimator.predict(X)
    elif hasattr(estimator, "fit_predict"):
        labels = estimator.fit_predict(X)
    else:
        estimator.fit(X)
        labels = getattr(estimator, "labels_", estimator.predict(X))
    elapsed = perf_counter() - start
    
    # If estimator is a pipeline, we need to transform X to the space used for clustering
    # to calculate internal metrics correctly.
    if isinstance(estimator, Pipeline):
        # Transform data using all steps except the last one (the clusterer)
        X_transformed = estimator[:-1].transform(X)
        if hasattr(X_transformed, "toarray"):
            X_transformed = X_transformed.toarray()
    else:
        X_transformed = X

    metrics = compute_internal_metrics(X_transformed, labels, noise_label=noise_label)
    metrics.update(compute_external_metrics(labels, reference))
    metrics.update(cluster_size_summary(labels, noise_label=noise_label))
    metrics["algorithm"] = name
    metrics["fit_time_sec"] = elapsed
    
    # Try to get inertia from the final estimator step
    final_step = estimator[-1] if isinstance(estimator, Pipeline) else estimator
    metrics["inertia"] = float(getattr(final_step, "inertia_", np.nan))
    
    metrics["noise_label"] = noise_label if noise_label is not None else np.nan
    return pd.Series(metrics), labels, estimator


def plot_pca_projection(
    X: np.ndarray,
    labels: np.ndarray,
    title: str,
    ax: Optional[plt.Axes] = None,
    palette: str = "tab10",
    alpha: float = 0.8,
) -> plt.Axes:
    pca = PCA(n_components=2, random_state=RANDOM_STATE)
    components = pca.fit_transform(X)
    frame = pd.DataFrame(components, columns=["PC1", "PC2"])
    frame["cluster"] = labels
    if ax is None:
        _, ax = plt.subplots(figsize=(8, 6))
    sns.scatterplot(data=frame, x="PC1", y="PC2", hue="cluster", palette=palette, alpha=alpha, ax=ax)
    ax.set_title(title)
    ax.legend(title="cluster", bbox_to_anchor=(1.02, 1), loc="upper left")
    return ax


def elbow_silhouette_curves(X: np.ndarray, k_values: Iterable[int]) -> Tuple[np.ndarray, np.ndarray]:
    inertias, silhouettes = [], []
    for k in k_values:
        model = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init="auto")
        labels = model.fit_predict(X)
        inertias.append(model.inertia_)
        silhouettes.append(silhouette_score(X, labels))
    return np.array(inertias), np.array(silhouettes)


def nearest_neighbor_distance_plot(X: np.ndarray, min_samples: int) -> np.ndarray:
    nn = NearestNeighbors(n_neighbors=min_samples)
    nn.fit(X)
    distances, _ = nn.kneighbors(X)
    distances = np.sort(distances[:, -1])
    return distances

---
# Task 2 · Preprocessing Guardrails (⏱️ ~30 min)

**Scenario:** Dollar magnitudes in credit datasets span 6+ orders, so naive clustering collapses to "everyone looks maxed out." We slow down to show how scaling, capping, and ratio features change the personas.

**Business Context:**
- Risk leadership rejects personas that ignore payment discipline or utilization. We need defensible feature engineering steps before any clustering bake-off.
- Students must see leakage warnings early—especially when we derive cluster labels and feed them into supervised lift models.

**You'll Learn:**
- When to combine scaling + encoding inside a single pipeline so numeric explosions do not dominate distance calculations.
- How log/ratio transforms calm the balance vs. purchase axis.
- How to brief students on diagnostics (elbow, silhouette, widgets) using credit-specific terminology.

## 2.1 Scaling & Encoding (Credit Portfolio)
Compare naive K-Means on raw dollar columns with a pipeline that scales every numeric column and one-hot encodes tenure/limit buckets.

In [ ]:
# TODO: Implement this task
raise NotImplementedError("Student exercise")


✅ **Checkpoint 2.1:** Highlight how preprocessing boosts cohesion/separation and surfaces marketing response structure.

## 2.2 Handling Skew & Imbalanced Axes (Balance vs. Purchases)
Show how log/ratio transformations tame balance magnitude so it no longer overwhelms spending behaviour.

In [ ]:
# TODO: Implement this task
raise NotImplementedError("Student exercise")


✅ **Checkpoint 2.2:** Ask students to quantify cluster size balance before/after scaling.

## 2.3 Choosing \(k\): Elbow and Silhouette Diagnostics

In [ ]:
# TODO: Implement this task
raise NotImplementedError("Student exercise")


✅ **Checkpoint 2.3:** Encourage learners to justify their chosen \(k\) using both curves.

## 2.4 Interactive Elbow Exploration
Give students agency to explore how the metrics shift when they move \(k\). In class, ask them to stop when they find a knee they can justify.

In [ ]:
try:
    import ipywidgets as widgets
    from IPython.display import display

    slider = widgets.IntSlider(value=5, min=2, max=12, step=1, description="k clusters")

    def explore_k(k: int):
        estimator = Pipeline([
            ("prep", segment_preprocessor),
            ("cluster", KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init="auto")),
        ])
        metrics, _, _ = evaluate_algorithm(
            f"K-Means (k={k})", estimator, segment_features, reference=None
        )
        cols = [
            "algorithm",
            "silhouette",
            "calinski_harabasz",
            "davies_bouldin",
            "fit_time_sec",
            "n_clusters",
            "inertia",
        ]
        display(metrics.to_frame().T[cols])

    widgets.interact(explore_k, k=slider)
except ImportError:
    print("Install ipywidgets to use the interactive elbow explorer (pip install ipywidgets).")

## 2.5 Dimensionality Reduction Inside vs Outside the Pipeline

In [ ]:
# TODO: Implement this task
raise NotImplementedError("Student exercise")


✅ **Checkpoint 2.5:** Discuss trade-offs between dimensionality reduction for speed vs. preserving variance for clustering.

---
# Task 3 · Algorithm Showdown (⏱️ ~35 min)

**Scenario:** With preprocessing dialed in, we stress-test multiple clustering families on the same credit-card feature space to showcase trade-offs.

**Business Context:**
- Model risk wants defensible reasoning for why one algorithm is chosen over another (speed vs. nuance vs. explainability).
- Students need to see how runtime, cluster counts, and interpretability shift per method when all features are financial ratios.

**You'll Learn:**
- How to benchmark centroid, density, probabilistic, spectral, and hierarchical approaches on the credit portfolio.
- How to narrate algorithm selection with both metrics and visuals tied to utilization and payment behaviour.

## 3.1 K-Means vs MiniBatch K-Means

In [ ]:
# TODO: Implement this task
raise NotImplementedError("Student exercise")


MiniBatch reaches similar quality with lower latency—point out its usefulness for streaming or very large datasets.

## 3.2 Density-Based Methods (DBSCAN & HDBSCAN)
Use the credit-card embeddings to illustrate epsilon selection, then compare DBSCAN with HDBSCAN on the same feature space.

In [ ]:
# TODO: Implement this task
raise NotImplementedError("Student exercise")


In [ ]:
# TODO: Implement this task
raise NotImplementedError("Student exercise")


Key messages: DBSCAN struggles with varying density; HDBSCAN adapts cluster counts and isolates noise responsibly.

## 3.3 Gaussian Mixture Models vs K-Means
Show soft assignments and compare likelihood-based modelling.

In [ ]:
# TODO: Implement this task
raise NotImplementedError("Student exercise")


Soft assignments help surface ambiguous customers—flag rows with max responsibility below ~0.6 for manual review.

## 3.4 Spectral and Agglomerative Clustering
Spectral handles non-convex shapes in the credit portfolio, while agglomerative provides hierarchical narratives for spend tiers.


In [ ]:
# TODO: Implement this task
raise NotImplementedError("Student exercise")


In [ ]:
# TODO: Implement this task
raise NotImplementedError("Student exercise")


Discuss linkage choices (Ward vs. average vs. complete) and when to expose dendrograms for interpretability.

## 3.5 Visualization with t-SNE
Project high-dimensional marketing embeddings into 2-D to sanity check cluster geometry.

---
# Task 4 · Clusters → Features & Personas (⏱️ ~30 min)

**Scenario:** After the algorithm bake-off, students must convert unsupervised clusters into supervised lift and a stakeholder-ready risk playbook.

**Business Context:**
- Credit policy wants leakage-free features they can trust inside delinquency or upsell models.
- Executives expect a concise persona playbook (utilization, payments, cash advance risk), not just scatter plots.

**You'll Learn:**
- How to add cluster memberships as supervised features without contaminating validation folds.
- How to summarise each persona with metrics and action ideas that resonate with risk, finance, and marketing.

In [ ]:
# TODO: Implement this task
raise NotImplementedError("Student exercise")


## 4.1 Clustering as Feature Engineering
Demonstrate how unsupervised learning can boost supervised model performance by capturing non-linear structure as a new feature.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.metrics import f1_score

# TODO: Implement this task
raise NotImplementedError("Student exercise")


## 4.2 Cluster Personas & Action Playbook
After running multiple algorithms, translate the credit-card clusters into something a stakeholder can act on. We will:
1. Fit our best-performing K-Means pipeline.
2. Summarise each cluster's utilization, payment behaviour, and cash-advance appetite.
3. Auto-generate a draft playbook with targeting or risk actions per persona.

In [ ]:
# TODO: Implement this task
raise NotImplementedError("Student exercise")


---
## Wrap-Up & Teaching Notes
- Reinforce why scaling + ratio engineering (utilization, payment-to-balance, purchase-to-limit) are prerequisites before comparing algorithms.
- Choose \(k\) using multiple diagnostics *and* business context (risk appetite, number of playbook slots).
- When density varies, prefer HDBSCAN or tune DBSCAN carefully; expect noise points and justify them as “outlier accounts.”
- Gaussian mixtures surface ambiguous accounts; leverage probabilities for queue prioritization or manual review.
- Spectral and agglomerative methods unlock non-spherical and hierarchical patterns—great discussion points once K-Means intuition is solid.
- Use t-SNE/PCA plots as sanity checks, not substitutes for quantitative metrics or business reasonableness checks.